### 第六周，第二天

我们即将创建并使用自己的 MCP Server 和 MCP Client！

这并不难，但也不是超级简单。MCP 的令人兴奋之处在于它让分享和使用其他 MCP Server 变得非常容易 — 但创建我们自己的 MCP Server 确实需要一些工作。

让我们回顾一下主要由辛勤工作的工程团队编写的 Python 代码：

accounts.py

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

In [ ]:
from accounts import Account

In [ ]:
account = Account.get("Ed")
account

In [ ]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

In [ ]:
account.report()

In [ ]:
account.list_transactions()

### 现在我们来编写一个 MCP 服务器并直接使用它！

In [ ]:
# 现在让我们把 accounts 服务器用作 MCP 服务器

params = {"command": "uv", "args": ["run", "accounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [ ]:
mcp_tools

In [ ]:
instructions = "你能够为客户管理账户，并回答关于账户的问题。"
request = "我叫 Ed，我的账户名是 Ed。我的余额和持仓情况如何？"
model = "gpt-4.1-mini"

In [ ]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


### 现在让我们构建自己的 MCP 客户端

In [ ]:
from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

In [ ]:
request = "我叫 Ed，我的账户名是 Ed。我的余额是多少？"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

In [ ]:
context = await read_accounts_resource("ed")
print(context)

In [ ]:
from accounts import Account
Account.get("ed").report()

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">练习</h2>
            <span style="color:#ff7800;">创建你自己的 MCP Server！编写一个简单的函数来返回当前日期，并将其暴露为工具，让 Agent 能够告诉你今天的日期。<br/>更难的可选练习：然后创建一个 MCP Client，并使用原生 OpenAI 调用（不使用 Agents SDK）通过你的客户端来使用你的工具。
            </span>
        </td>
    </tr>
</table>